This notebook can be used to detect cars in images using YOLO. First we fine tune an Ultralytics YOLO model using the Stanford Cars dataset. Then we predict cars using the COCO dataset.

Mount disk access from Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Before running this notebook, Stanford Cars (https://www.kaggle.com/datasets/eduardo4jesus/stanford-cars-dataset) was downloaded and unzipped in Google Drive. The below commented code formats the raw archive so that it can be used to fine tune YOLO models from Ultralytics. We use a train/valid/test split of 80/10/10 with 6000 training and 600 validation/testing images.

In [ ]:
# import os
# import random
# import shutil
# from scipy.io import loadmat
# import cv2

# BASE_DIR = '/content/drive/MyDrive/stanfordcars/archive'
# IMAGES_DIR = os.path.join(BASE_DIR, 'cars_train/cars_train')
# ANNOT_DIR = os.path.join(BASE_DIR, 'car_devkit/devkit/cars_train_annos.mat')
# OUTPUT_DIR = '/content/drive/MyDrive/stanfordcars_yolo'

# os.makedirs(OUTPUT_DIR, exist_ok=True)
# subdirs = ['train', 'val', 'test']
# for s in subdirs:
#     os.makedirs(f'{OUTPUT_DIR}/images/{s}', exist_ok=True)
#     os.makedirs(f'{OUTPUT_DIR}/labels/{s}', exist_ok=True)

In [ ]:
# raw_annos = loadmat(ANNOT_DIR)['annotations'][0]
# annos = []

# for a in raw_annos:
#     fname = a[5][0]
#     src_img_path = os.path.join(IMAGES_DIR, fname)
#     img = cv2.imread(src_img_path)
#     annos.append((a, img.shape[:2]))

# if len(annos) < 7200:
#     raise ValueError(f'Not enough images: {len(annos)}')

# random.seed(42)
# annos = random.sample(list(annos), 7200)
# splits = {
#     'train': annos[:6000],
#     'val': annos[6000:6600],
#     'test': annos[6600:]
# }

In [ ]:
# def convert_yolo_bbox(x1, y1, x2, y2, img_w, img_h):
#     x_center = (x1 + x2) / 2.0 / img_w
#     y_center = (y1 + y2) / 2.0 / img_h
#     width = (x2 - x1) / img_w
#     height = (y2 - y1) / img_h
#     return f'0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}'

# for split_name, split_annos in splits.items():
#     for a, (h, w) in split_annos:
#         fname = a[5][0]
#         x1, y1, x2, y2 = [int(a[i][0]) for i in range(0, 4)]
#         src_img_path = os.path.join(IMAGES_DIR, fname)
#         dst_img_path = os.path.join(OUTPUT_DIR, f'images/{split_name}/{fname}')
#         label_path = os.path.join(OUTPUT_DIR, f'labels/{split_name}/{fname.replace(".jpg", ".txt")}')

#         # Write label
#         with open(label_path, 'w') as f:
#             f.write(convert_yolo_bbox(x1, y1, x2, y2, w, h))

#         # Write image
#         shutil.copy(src_img_path, dst_img_path)

<ipython-input-117-9ca0c4739bc5>:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x1, y1, x2, y2 = [int(a[i][0]) for i in range(0, 4)]


Ultralytics (https://docs.ultralytics.com/) maintains all versions of YOLO and we'll use it to build YOLO models.

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 34.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

Write an Ultralytics configuration file that is used to fine tune a YOLO model using Stanford Cars. Our use case only wants to detect one class: car. After creating the configuration file, the below commented code writes it to disk in Google Drive.

In [ ]:
%%writefile yolo.yaml
path: /content/drive/MyDrive/stanfordcars_yolo
train: images/train
val: images/val
nc: 1
names: ['car']

Writing yolo.yaml


In [ ]:
# !mkdir -p /content/drive/MyDrive/stanfordcars_yolo/configs
# !cp yolo.yaml /content/drive/MyDrive/stanfordcars_yolo/configs/

Fine tune a YOLO model using Stanford Cars!

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')
results = model.train(
    data='/content/drive/MyDrive/stanfordcars_yolo/configs/yolo.yaml',
    epochs=10,
    batch=16,
    imgsz=640,
    name='yolo11m_cars',
    project='/content/drive/MyDrive/stanfordcars_yolo'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 38.8M/38.8M [00:00<00:00, 107MB/s] 


Ultralytics 8.3.150 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/stanfordcars_yolo/configs/yolo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11m_cars, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0

100%|██████████| 755k/755k [00:00<00:00, 40.9MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              


...

      10/10      8.08G     0.3612     0.1961     0.9942         15        640: 100%|██████████| 375/375 [03:32<00:00,  1.77it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:07<00:00,  2.39it/s]

                   all        600        600      0.992          1      0.995      0.942



10 epochs completed in 1.276 hours.
Optimizer stripped from /content/drive/MyDrive/stanfordcars_yolo/yolo11m_cars/weights/last.pt, 40.5MB
Optimizer stripped from /content/drive/MyDrive/stanfordcars_yolo/yolo11m_cars/weights/best.pt, 40.5MB

Validating /content/drive/MyDrive/stanfordcars_yolo/yolo11m_cars/weights/best.pt...
Ultralytics 8.3.150 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:09<00:00,  1.90it/s]


                   all        600        600      0.992          1      0.995      0.941
Speed: 0.2ms preprocess, 7.3ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/drive/MyDrive/stanfordcars_yolo/yolo11m_cars


Evaluate the fine tuned YOLO model. We reload the best weights from fine tuning from disk.

In [5]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/stanfordcars_yolo/yolo11m_cars/weights/best.pt')
metrics = model.val()

Ultralytics 8.3.150 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 22.2MB/s]


val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 0.4±0.6 MB/s, size: 223.0 KB)


val: Scanning /content/drive/MyDrive/stanfordcars_yolo/labels/val.cache... 600 images, 0 backgrounds, 0 corrupt: 100%|██████████| 600/600 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 38/38 [00:16<00:00,  2.35it/s]


                   all        600        600      0.998      0.995      0.995       0.94
Speed: 1.0ms preprocess, 16.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs/detect/val


Download and unzip the COCO train images and annotations. We'll make predictions using our fine tuned YOLO model. The commented code below formats the images and annotations so that they can be used by Ultralytics.

In [ ]:
# !wget http://images.cocodataset.org/zips/train2017.zip
# !unzip -q train2017.zip -d ./

In [12]:
# import os
# import shutil

# image_src_dir = '/content/drive/MyDrive/coco/images/train2017'
# label_src_dir = '/content/drive/MyDrive/coco/labels/train2017'
# image_dst_dir = '/content/drive/MyDrive/coco/images/train10k'
# label_dst_dir = '/content/drive/MyDrive/coco/labels/train10k'

# os.makedirs(image_dst_dir, exist_ok=True)
# os.makedirs(label_dst_dir, exist_ok=True)

# image_files = sorted([f for f in os.listdir(image_src_dir)])[:10000]

# for img_file in image_files:
#     base_name = os.path.splitext(img_file)[0]
#     label_file = base_name + '.txt'
#     label_path = os.path.join(label_src_dir, label_file)

#     if os.path.exists(label_path):
#         shutil.copy2(os.path.join(image_src_dir, img_file), os.path.join(image_dst_dir, img_file))
#         shutil.copy2(label_path, os.path.join(label_dst_dir, label_file))

Perform car detection using our fine tuned YOLO model. We reload the best weights from fine tuning from disk.

In [20]:
import os
import shutil
from ultralytics import YOLO

image_dir = '/content/drive/MyDrive/coco/images/train10k'
label_dir = '/content/drive/MyDrive/coco/labels/train10k'
output_img_dir = '/content/drive/MyDrive/coco/car_only_train10k/images'
output_lbl_dir = '/content/drive/MyDrive/coco/car_only_train10k/labels'

os.makedirs(output_img_dir, exist_ok=True)
os.makedirs(output_lbl_dir, exist_ok=True)

model = YOLO('/content/drive/MyDrive/stanfordcars_yolo/yolo11m_cars/weights/best.pt')

img_files = [f for f in os.listdir(image_dir)]

for fname in img_files:
    img_path = os.path.join(image_dir, fname)
    lbl_path = os.path.join(label_dir, fname.replace('.jpg', '.txt'))
    results = model.predict(source=img_path, conf=0.9, verbose=False)
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        class_ids = boxes.cls.cpu().numpy().astype(int)
        if 0 in class_ids:
            shutil.copy(img_path, os.path.join(output_img_dir, fname))
            shutil.copy(lbl_path, os.path.join(output_lbl_dir, os.path.basename(lbl_path)))